
# Week 4 — Classification Report

**Dataset:** `weather_classification_data.csv`  
**Objective:** Clean, analyze, visualize, preprocess the data, and train 2–3 classification models to predict **Weather Type** (Rainy, Sunny, Cloudy, Snowy).  
**Libraries:** NumPy, Pandas, Matplotlib (for plots), scikit-learn (for preprocessing & models).

## Checklist
- [x] Data loading & preview
- [x] Missing values handling (with justification)
- [x] Categorical encoding & numeric scaling
- [x] Train/test split (80/20)
- [x] EDA & Matplotlib visualizations (histograms, scatter, box, correlation heatmap)
- [x] Classification models (Logistic Regression, Decision Tree, Random Forest)
- [x] Evaluation (Accuracy, Precision, Recall, F1, Confusion Matrix)
- [x] Comparison table & discussion


In [ ]:

# Imports & Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Ensure plots render inline
# (In JupyterLab/Notebook this is automatic; included for clarity.)
# %matplotlib inline

DATA_DIR = Path("..") / "data"
CSV_PATH = DATA_DIR / "weather_classification_data.csv"

# Load data
df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
df.head()



## 1) Data Cleaning & Preparation

### 1.1 Missing Values
We identify missing values and handle them as follows:
- **Numeric features:** impute with median (robust to outliers).
- **Categorical features:** impute with most frequent (mode) to preserve common categories.

**Justification:** The dataset intentionally includes outliers (e.g., humidity >100%, very high wind speed). Median imputation is robust to skew/outliers compared to mean. For categories, most frequent imputation avoids introducing synthetic categories while keeping pipelines simple.


In [ ]:

# Basic info & NA summary
display(df.info())
na_counts = df.isna().sum().sort_values(ascending=False)
na_counts



### 1.2 Feature Types & Target
We'll treat **Weather Type** as the target. Categorical features will be one-hot encoded. Numeric features will be standardized.

> Note: Some numeric columns may include out-of-range values by design (synthetic). We'll scale but not clip to keep the outlier signal intact for tree-based models. Logistic regression can still benefit from scaling.


In [ ]:

# Identify target and feature types
target_col = "Weather Type"
assert target_col in df.columns, "Target 'Weather Type' not found!"

X = df.drop(columns=[target_col])
y = df[target_col]

numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_cols, categorical_cols


In [ ]:

# 1.3 Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

len(X_train), len(X_test)



## 2) Exploratory Data Analysis (EDA) & Visualization

We explore distributions, relationships, and correlations using Matplotlib (no seaborn). All plots are labeled and readable.


In [ ]:

# Summary statistics
display(df.describe(include='all'))

# Histograms for numeric features
for col in numeric_cols:
    plt.figure()
    plt.hist(df[col].dropna(), bins=30)
    plt.title(f"Histogram of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.show()

# Box plots for numeric features
for col in numeric_cols:
    plt.figure()
    plt.boxplot(df[col].dropna(), vert=True)
    plt.title(f"Box plot of {col}")
    plt.ylabel(col)
    plt.show()

# Scatter plots for a few numeric pairs (limit to first 4 pairs to keep concise)
pairs = []
for i in range(min(4, len(numeric_cols)-1)):
    pairs.append((numeric_cols[i], numeric_cols[i+1]))

for xcol, ycol in pairs:
    plt.figure()
    plt.scatter(df[xcol], df[ycol])
    plt.title(f"Scatter: {xcol} vs {ycol}")
    plt.xlabel(xcol)
    plt.ylabel(ycol)
    plt.show()

# Correlation heatmap for numeric features
if len(numeric_cols) > 1:
    corr = df[numeric_cols].corr(numeric_only=True)
    plt.figure(figsize=(8,6))
    plt.imshow(corr, interpolation='nearest')
    plt.title("Correlation Heatmap (numeric features)")
    plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=90)
    plt.yticks(range(len(numeric_cols)), numeric_cols)
    plt.colorbar()
    plt.tight_layout()
    plt.show()



## 3) Modeling (Classification Only)

We compare three complementary classifiers:

1. **Logistic Regression (multinomial):** A strong linear baseline; fast and interpretable; benefits from scaled features.
2. **Decision Tree:** Captures non-linearities and interactions; robust to monotonic transformations; interpretable structure.
3. **Random Forest:** An ensemble of trees reducing variance; often strong out-of-the-box on tabular data with mixed feature types.

We build a common preprocessing pipeline (imputation + encoding + scaling) and plug different estimators.


In [ ]:

from sklearn.impute import SimpleImputer

# Preprocessing for numeric and categorical columns
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols),
])

# Define models
models = {
    "LogReg": LogisticRegression(max_iter=1000, multi_class="multinomial"),
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "RandomForest": RandomForestClassifier(n_estimators=300, random_state=42)
}

results = []

for name, est in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocessor),
                          ("model", est)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
    rec = recall_score(y_test, y_pred, average="macro", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
    results.append({"Model": name, "Accuracy": acc, "Precision(macro)": prec, "Recall(macro)": rec, "F1(macro)": f1})

results_df = pd.DataFrame(results).sort_values("F1(macro)", ascending=False)
display(results_df)

best_name = results_df.iloc[0]["Model"]
print("Best by F1(macro):", best_name)


In [ ]:

# Confusion matrices & classification reports for each model
for name, est in models.items():
    pipe = Pipeline(steps=[("preprocess", preprocessor),
                          ("model", est)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    print(f"=== {name} ===")
    print(classification_report(y_test, y_pred, zero_division=0))
    cm = confusion_matrix(y_test, y_pred, labels=sorted(y_test.unique()))
    
    plt.figure(figsize=(6,5))
    plt.imshow(cm, interpolation='nearest')
    plt.title(f"Confusion Matrix — {name}")
    tick_labels = sorted(y_test.unique())
    plt.xticks(range(len(tick_labels)), tick_labels, rotation=45)
    plt.yticks(range(len(tick_labels)), tick_labels)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.colorbar()
    plt.tight_layout()
    plt.show()


In [ ]:

# Visual comparison of model F1 scores
plt.figure()
plt.bar(results_df["Model"], results_df["F1(macro)"])
plt.title("Model Comparison — F1(macro)")
plt.xlabel("Model")
plt.ylabel("F1(macro)")
plt.show()



## 4) Discussion & Conclusion

- **Best model:** Based on F1(macro), shown above. Random Forest often performs well on tabular data thanks to non-linear splits and ensembling.
- **Why it may win:** Handles outliers and complex interactions; less sensitive to feature scaling; robust to irrelevant features due to averaging across trees.
- **Future Work:** Hyperparameter tuning (e.g., depth, min_samples_split for trees/forest; C & penalty for logistic regression), calibration for probabilistic outputs, and feature engineering (e.g., interaction terms, derived meteorological indices).

---

**How to Submit:** Ensure this notebook runs end-to-end. Commit the entire `week_4` directory (including `classification/` and `data/`) to your GitHub repository.
